In [14]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [15]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [16]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [17]:
# Add Compounds
sim.AddCompound("Water")

In [18]:
one = sim.AddFlowsheetObject('Material Stream','Stream-1')
two = sim.AddFlowsheetObject('Material Stream','Stream-2')
energy_stream = sim.AddFlowsheetObject('Energy Stream','energy_stream')
PS_1 = sim.AddFlowsheetObject('Pipe Segment','Pipe-1')

In [19]:
one = one.GetAsObject()
two = two.GetAsObject()
energy_stream = energy_stream.GetAsObject()
PS_1 = PS_1.GetAsObject()

In [20]:
sim.ConnectObjects(one.GraphicObject, PS_1.GraphicObject,  -1, -1)
sim.ConnectObjects(PS_1.GraphicObject, two.GraphicObject,  -1, -1)
sim.ConnectObjects(PS_1.GraphicObject, energy_stream.GraphicObject, -1, -1)

In [21]:
sim.AutoLayout()

In [22]:
    # property package
    Thermo_Package = sim.CreateAndAddPropertyPackage("Steam Tables (IAPWS-IF97)")

In [23]:
# Setting the inlet stream conditions
one.SetPressure(101325)
one.SetTemperature(300)
one.SetMassFlow(1)
one.SetOverallComposition(Array[float]([1.0]))

# Setting pipe segements parameters
PS_1.Specmode = PS_1.Specmode.Length         # Set spec mode to Length
PS_1.SelectedFlowPackage = PS_1.SelectedFlowPackage.Beggs_Brill  # Set flow package
PS_1.OutletPressure = 500000
PS_1.OutletTemperature = 298.15
PS_1.TolP = 100
PS_1.TolT = 0.01
# TODO Joule Thomson effect is not working, investigate
PS_1.SlurryViscosityMode = 0

# 1. Create a new PipeSection object
from DWSIM.UnitOperations.UnitOperations.Auxiliary.Pipe import PipeSection, PipeEditorStatus

# Build the section
ps = PipeSection()
ps.Indice = 1
ps.TipoSegmento = "Straight Tube Section"
ps.Material = "Stainless Steel"
ps.PipeWallRugosity = 4.5e-5
ps.PipeWallThermalConductivityExpression = "14.941"
ps.Incrementos = 10
ps.Quantidade = 1
ps.Comprimento = 100.0
ps.Elevacao = 0.0
ps.DE = 60.325 / 25.4
ps.DI = 52.5018 / 25.4

# ✅ Work directly on the backing field m_profile — bypass the property getter copy issue
PS_1.m_profile.Sections.Clear()
PS_1.m_profile.Sections.Add(1, ps)
PS_1.m_profile.Status = PipeEditorStatus.OK

# Verify it actually persisted on the real backing field
print(f"Sections count: {PS_1.Profile.Sections.Count}")
print(f"Key used: {list(PS_1.Profile.Sections.Keys)}")   # should show [1]
print(f"Profile status: {PS_1.Profile.Status}")
# Both counts and statuses should now match

Sections count: 1
Key used: [1]
Profile status: OK


In [24]:
# Inspect PS_1 for any flags/properties related to profile validation
print(dir(PS_1))

# Specifically look for anything like:
print(f"HasProfile: {getattr(PS_1, 'HasProfile', 'NOT FOUND')}")
print(f"ProfileOK: {getattr(PS_1, 'ProfileOK', 'NOT FOUND')}")
print(f"HydraulicProfileDefined: {getattr(PS_1, 'HydraulicProfileDefined', 'NOT FOUND')}")
print(f"UseUserDefinedProfile: {getattr(PS_1, 'UseUserDefinedProfile', 'NOT FOUND')}")

# Also check Profile object flags
print(f"\nProfile props: {dir(PS_1.Profile)}")
print(f"Profile.Sections type: {type(PS_1.Profile.Sections)}")

['AccumulationStream', 'AddDynamicProperty', 'AddExtraProperty', 'AdjustVarType', 'Annotation', 'AppendDebugLine', 'AttachedAdjustId', 'AttachedSpecId', 'AttachedUtilities', 'CALCT2', 'CalcOverallHeatTransferCoefficient', 'Calculate', 'CalculateEquilibrium', 'CalculateEquilibriumIntervalInSteps', 'Calculated', 'CalculationRoutineOverride', 'CanUsePreviousResults', 'CheckDirtyStatus', 'CheckSpec', 'ClassId', 'ClearExtraProperties', 'ClearPropertyPackageInstance', 'Clone', 'CloneJSON', 'CloneXML', 'CloseDynamicsEditForm', 'CloseEditForm', 'ComponentDescription', 'ComponentName', 'ConnectEnergyStream', 'ConnectFeedEnergyStream', 'ConnectFeedMaterialStream', 'ConnectProductEnergyStream', 'ConnectProductMaterialStream', 'CopyDataToClipboard', 'CreateChartAction', 'CreateDimensionsList', 'CreateDynamicProperties', 'CreateNew', 'DeCalculate', 'DebugMode', 'DebugText', 'DeltaP', 'DeltaQ', 'DeltaT', 'DetailedDebugReport', 'Dimensions', 'DisplayDynamicsEditForm', 'DisplayEditForm', 'DisplayExtra

In [25]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [26]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/18 Pipe Segment/18 Pipe Segment Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)